# Model Optimization: Knowledge Distillation

In this notebook, we'll explore knowledge distillation, a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model. This allows us to create models that are much smaller and faster while retaining most of the accuracy of the original model.

## 1. Import Dependencies

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DistilBertForSequenceClassification, DistilBertConfig
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

# Import utility functions
from utils import measure_inference_time, get_model_size, measure_memory_usage, plot_comparison

## 2. Load Baseline Metrics and Model Information

In [ ]:
# Load baseline metrics
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

# Load model information
with open('model_info.json', 'r') as f:
    model_info = json.load(f)

print(f"Loaded baseline metrics for {len(baseline_metrics)} models")

## 3. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}

## 4. Create Functions for Loading Models and Preparing Inputs

In [ ]:
def load_model(model_key):
    """Load model and tokenizer from local path."""
    model_data = model_info[model_key]
    model_path = model_data["local_path"]
    task = model_data["task"]
    
    print(f"Loading {model_data['model_name']} for {task}...")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Load model based on task
    if task == "sequence-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
    else:
        raise ValueError(f"Knowledge distillation example only supports sequence-classification task")
    
    # Move model to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()  # Set model to evaluation mode
    
    return model, tokenizer, device

def prepare_inputs(model_key, tokenizer, device):
    """Prepare inputs for the model based on task."""
    task = model_info[model_key]["task"]
    sample_input = sample_inputs[model_key]
    
    if task == "sequence-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    else:
        raise ValueError(f"Knowledge distillation example only supports sequence-classification task")
    
    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    return inputs

## 5. Select a Model for Knowledge Distillation

For this example, we'll use a BERT model for sentiment analysis as our teacher model and create a smaller DistilBERT model as our student.

In [ ]:
# Select a model for knowledge distillation (sentiment analysis)
model_key = "sentiment_analysis"

# Check if the selected model is available
if model_key not in model_info:
    raise ValueError(f"Model {model_key} not found in model_info.json")

# Check if the task is supported
if model_info[model_key]["task"] != "sequence-classification":
    raise ValueError(f"Knowledge distillation example only supports sequence-classification task")

print(f"Selected model: {model_info[model_key]['model_name']}")
print(f"Task: {model_info[model_key]['task']}")

## 6. Load Dataset for Distillation

We'll use the SST-2 dataset for sentiment analysis distillation.

In [ ]:
# Load SST-2 dataset
dataset = load_dataset("glue", "sst2")
print(f"Dataset loaded: {dataset}")

# Display a few examples
print("\nSample examples:")
for i in range(3):
    print(f"Example {i+1}: {dataset['train'][i]}")

## 7. Prepare Dataset for Distillation

In [ ]:
# Load teacher model
teacher_model, tokenizer, device = load_model(model_key)

# Function to tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True)

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
tokenized_dataset = tokenized_dataset.remove_columns(["sentence", "idx"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

print(f"Tokenized dataset: {tokenized_dataset}")

## 8. Generate Soft Labels from Teacher Model

In [ ]:
# Function to generate soft labels from teacher model
def generate_soft_labels(batch):
    inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
    with torch.no_grad():
        outputs = teacher_model(**inputs)
        logits = outputs.logits
        soft_labels = F.softmax(logits, dim=1)
    
    return {"soft_labels": soft_labels.cpu()}

# Generate soft labels for training set
print("Generating soft labels from teacher model...")
train_dataset = tokenized_dataset["train"].map(
    generate_soft_labels,
    batched=True,
    batch_size=16
)

print("Soft labels generated")

## 9. Create Student Model

In [ ]:
# Create student model (DistilBERT)
config = DistilBertConfig.from_pretrained(
    "distilbert-base-uncased",
    num_labels=teacher_model.config.num_labels
)

student_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    config=config
)

# Move student model to the same device as teacher
student_model = student_model.to(device)

print(f"Student model created: {student_model.__class__.__name__}")

## 10. Define Distillation Loss Function

In [ ]:
# Define distillation loss function
class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, alpha=0.5, temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.alpha = alpha  # Weight for distillation loss
        self.temperature = temperature  # Temperature for softening probability distributions
    
    def compute_loss(self, model, inputs, return_outputs=False):
        # Extract soft labels
        soft_labels = inputs.pop("soft_labels", None)
        
        # Standard cross-entropy loss
        outputs = model(**inputs)
        student_logits = outputs.logits
        loss_ce = F.cross_entropy(student_logits, inputs["labels"])
        
        # Distillation loss
        if soft_labels is not None:
            loss_kd = F.kl_div(
                F.log_softmax(student_logits / self.temperature, dim=-1),
                F.softmax(soft_labels / self.temperature, dim=-1),
                reduction="batchmean"
            ) * (self.temperature ** 2)
            
            # Combine losses
            loss = (1 - self.alpha) * loss_ce + self.alpha * loss_kd
        else:
            loss = loss_ce
        
        return (loss, outputs) if return_outputs else loss

## 11. Train Student Model

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# Create trainer
trainer = DistillationTrainer(
    model=student_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=tokenized_dataset["validation"],
    teacher_model=teacher_model,
    alpha=0.5,
    temperature=2.0,
)

# Train student model
print("Training student model...")
trainer.train()
print("Training complete")

## 12. Save Distilled Model

In [ ]:
# Create directory for distilled model
model_name = model_info[model_key]["model_name"].replace("/", "_")
distilled_dir = os.path.join("models", f"{model_name}_distilled")
os.makedirs(distilled_dir, exist_ok=True)

# Save distilled model
student_model.save_pretrained(distilled_dir)
tokenizer.save_pretrained(distilled_dir)

print(f"Distilled model saved to {distilled_dir}")

## 13. Evaluate Distilled Model

In [ ]:
# Evaluate on validation set
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

## 14. Compare Teacher and Student Models

In [ ]:
# Prepare inputs for comparison
inputs = prepare_inputs(model_key, tokenizer, device)

# Measure teacher model metrics
teacher_size = get_model_size(teacher_model)
teacher_inference_time = measure_inference_time(teacher_model, inputs)
teacher_memory_usage = measure_memory_usage(teacher_model, inputs)

# Measure student model metrics
student_size = get_model_size(student_model)
student_inference_time = measure_inference_time(student_model, inputs)
student_memory_usage = measure_memory_usage(student_model, inputs)

# Print comparison
print("Model Comparison:")
print(f"{'Metric':<20} {'Teacher':<15} {'Student':<15} {'Reduction (%)':<15}")
print("-" * 65)
print(f"{'Size (MB)':<20} {teacher_size:<15.2f} {student_size:<15.2f} {(1 - student_size/teacher_size)*100:<15.2f}")
print(f"{'Inference Time (ms)':<20} {teacher_inference_time:<15.2f} {student_inference_time:<15.2f} {(1 - student_inference_time/teacher_inference_time)*100:<15.2f}")
print(f"{'Memory Usage (MB)':<20} {teacher_memory_usage:<15.2f} {student_memory_usage:<15.2f} {(1 - student_memory_usage/teacher_memory_usage)*100:<15.2f}")

In [ ]:
# Create comparison data for visualization
comparison_data = [
    {"Model": "Teacher", "Metric": "Size (MB)", "Value": teacher_size},
    {"Model": "Student", "Metric": "Size (MB)", "Value": student_size},
    {"Model": "Teacher", "Metric": "Inference Time (ms)", "Value": teacher_inference_time},
    {"Model": "Student", "Metric": "Inference Time (ms)", "Value": student_inference_time},
    {"Model": "Teacher", "Metric": "Memory Usage (MB)", "Value": teacher_memory_usage},
    {"Model": "Student", "Metric": "Memory Usage (MB)", "Value": student_memory_usage}
]

comparison_df = pd.DataFrame(comparison_data)

# Plot comparisons
plt.figure(figsize=(15, 5))

for i, metric in enumerate(["Size (MB)", "Inference Time (ms)", "Memory Usage (MB)"]):
    plt.subplot(1, 3, i+1)
    metric_data = comparison_df[comparison_df["Metric"] == metric]
    sns.barplot(x="Model", y="Value", data=metric_data)
    plt.title(metric)
    plt.ylabel(metric)

plt.tight_layout()
plt.show()

## 15. Save Distilled Model Metrics

In [ ]:
# Save distilled model metrics
distilled_metrics = {
    model_key: {
        "model_key": model_key,
        "model_name": model_info[model_key]["model_name"],
        "task": model_info[model_key]["task"],
        "model_size": student_size,
        "inference_time": student_inference_time,
        "memory_usage": student_memory_usage,
        "distilled_path": distilled_dir,
        "teacher_model": model_info[model_key]["model_name"],
        "teacher_size": teacher_size,
        "teacher_inference_time": teacher_inference_time,
        "teacher_memory_usage": teacher_memory_usage,
        "size_reduction": (1 - student_size/teacher_size) * 100,
        "speed_improvement": (1 - student_inference_time/teacher_inference_time) * 100,
        "memory_reduction": (1 - student_memory_usage/teacher_memory_usage) * 100,
        "eval_results": eval_results
    }
}

# Save metrics to file
with open('distilled_metrics.json', 'w') as f:
    json.dump(distilled_metrics, f, indent=2)

print("Distilled model metrics saved to distilled_metrics.json")

## 16. Estimate Cost Savings

In [ ]:
def estimate_monthly_cost(inference_time_ms, model_size_mb, requests_per_month=1000000):
    """Estimate monthly cost for running a model in production."""
    # Assumptions
    compute_cost_per_hour = 0.5  # $0.5 per hour for compute (e.g., ml.g4dn.xlarge)
    storage_cost_per_gb_month = 0.023  # $0.023 per GB-month for S3
    
    # Calculate compute cost
    inference_time_hours = (inference_time_ms * requests_per_month) / (1000 * 60 * 60)
    compute_cost = inference_time_hours * compute_cost_per_hour
    
    # Calculate storage cost
    storage_cost = (model_size_mb / 1024) * storage_cost_per_gb_month
    
    # Total cost
    total_cost = compute_cost + storage_cost
    
    return {
        "compute_cost": compute_cost,
        "storage_cost": storage_cost,
        "total_cost": total_cost
    }

In [ ]:
# Estimate costs for teacher and student models
teacher_cost = estimate_monthly_cost(teacher_inference_time, teacher_size)
student_cost = estimate_monthly_cost(student_inference_time, student_size)

# Calculate savings
monthly_savings = teacher_cost["total_cost"] - student_cost["total_cost"]
savings_percentage = (monthly_savings / teacher_cost["total_cost"]) * 100

# Print cost comparison
print("Cost Comparison (1M requests/month):")
print(f"{'Cost Component':<20} {'Teacher':<15} {'Student':<15} {'Savings':<15}")
print("-" * 65)
print(f"{'Compute Cost ($)':<20} {teacher_cost['compute_cost']:<15.2f} {student_cost['compute_cost']:<15.2f} {teacher_cost['compute_cost'] - student_cost['compute_cost']:<15.2f}")
print(f"{'Storage Cost ($)':<20} {teacher_cost['storage_cost']:<15.2f} {student_cost['storage_cost']:<15.2f} {teacher_cost['storage_cost'] - student_cost['storage_cost']:<15.2f}")
print(f"{'Total Cost ($)':<20} {teacher_cost['total_cost']:<15.2f} {student_cost['total_cost']:<15.2f} {monthly_savings:<15.2f}")
print(f"\nMonthly Savings: ${monthly_savings:.2f} ({savings_percentage:.2f}%)")

In [ ]:
# Create cost data for visualization
cost_data = [
    {"Model": "Teacher", "Cost Type": "Compute Cost", "Cost ($)": teacher_cost["compute_cost"]},
    {"Model": "Student", "Cost Type": "Compute Cost", "Cost ($)": student_cost["compute_cost"]},
    {"Model": "Teacher", "Cost Type": "Storage Cost", "Cost ($)": teacher_cost["storage_cost"]},
    {"Model": "Student", "Cost Type": "Storage Cost", "Cost ($)": student_cost["storage_cost"]}
]

cost_df = pd.DataFrame(cost_data)

# Plot cost comparison
plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="Cost ($)", hue="Cost Type", data=cost_df)
plt.title("Monthly Cost Comparison (1M requests/month)")
plt.ylabel("Cost ($)")
plt.tight_layout()
plt.show()

## 17. Next Steps

In this notebook, we've applied knowledge distillation to create a smaller, faster model that retains most of the accuracy of the original model. We've seen significant reductions in model size, inference time, and memory usage, leading to substantial cost savings.

In the next notebook, we'll deploy our models to AWS SageMaker and compare their performance in a production environment.